In [39]:
#run under conda deepchem environment includeing rdkit, smilesrnn, molscore and reinvent
import random
import os
import rdkit
from rdkit import Chem
import mols2grid

In [32]:
!pwd

/mnt/d/blog_in_progress/smiles_rnn_test


In [33]:
def split_smi_file(input_file, train_file, valid_file, ratio=0.75):
    """
    Randomly splits a .smi file into train and test files.
    """
    if not os.path.exists(input_file):
        print(f"File not found: {input_file}")
        return

    with open(input_file, 'r') as f:
        lines = f.readlines()
    
    # Remove empty lines
    lines = [line for line in lines if line.strip()]
    
    # Shuffle lines
    random.seed(42) # For reproducibility
    random.shuffle(lines)
    
    split_point = int(len(lines) * ratio)
    train_data = lines[:split_point]
    valid_data = lines[split_point:]
    
    with open(train_file, 'w') as f:
        f.writelines(train_data)
        
    with open(valid_file, 'w') as f:
        f.writelines(valid_data)
        
    print(f"Total lines: {len(lines)}")
    print(f"Train set ({len(train_data)} lines) saved to {train_file}")
    print(f"Valid set ({len(valid_data)} lines) saved to {valid_file}")

# Example usage:
split_smi_file('all_unique_processed_dataset.smi', 'train_dataset.smi', 'valid_dataset.smi', 0.75)

Total lines: 4323
Train set (3242 lines) saved to train_dataset.smi
Valid set (1081 lines) saved to valid_dataset.smi


In [34]:
!python /home/haolan/SMILES-RNN/scripts/train_prior.py \
-i train_dataset.smi \
-o trainprior \
-s trainprior \
--randomize \
--valid_smiles valid_dataset.smi \
RNN

Device set to cuda
Loading smiles
Randomizing 3242 training smiles
100%|█████████████████████████████████████| 3242/3242 [00:02<00:00, 1096.14it/s]
Returned 31789 randomized training smiles
Creating vocabulary
Creating vocabulary: 100%|████████████| 32870/32870 [00:00<00:00, 490817.73it/s]
Loading model
Beginning training
Epoch 1
100%|█████████████████████████████████████████| 257/257 [00:28<00:00,  9.06it/s]
Epoch 2
100%|█████████████████████████████████████████| 257/257 [00:24<00:00, 10.41it/s]
Epoch 3
100%|█████████████████████████████████████████| 257/257 [00:19<00:00, 12.89it/s]
Epoch 4
100%|█████████████████████████████████████████| 257/257 [00:24<00:00, 10.60it/s]
Epoch 5
100%|█████████████████████████████████████████| 257/257 [00:25<00:00, 10.24it/s]


In [35]:
%load_ext tensorboard
# Use absolute path and bind to localhost to ensure VS Code can proxy the port correctly
log_dir = os.path.abspath("trainprior/tb_trainprior")
print(f"Launching TensorBoard for: {log_dir}")
%tensorboard --logdir "{log_dir}" --host localhost

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
Launching TensorBoard for: /mnt/d/blog_in_progress/smiles_rnn_test/trainprior/tb_trainprior


Reusing TensorBoard on port 6014 (pid 1830901), started 0:57:25 ago. (Use '!kill 1830901' to kill it.)

In [38]:
!python /home/haolan/SMILES-RNN/scripts/sample_model.py -p trainprior/Prior_trainprior_Epoch-5.ckpt \
-m RNN \
-o sampleprior/Prior_100.smi \
-n 100 

Device set to cuda
Saving 100 smiles


In [44]:
def get_fresh_mols(sample_file, training_file):
    # Load training data
    training_smiles = set()
    with open(training_file, "r") as f:
        for line in f:
            if line.strip():
                mol = Chem.MolFromSmiles(line.split()[0])
                if mol: training_smiles.add(Chem.MolToSmiles(mol))
    
    # Filter sampled molecules
    fresh_mols = []
    with open(sample_file, "r") as f:
        for line in f:
            if line.strip():
                mol = Chem.MolFromSmiles(line.split()[0])
                if mol and Chem.MolToSmiles(mol) not in training_smiles:
                    fresh_mols.append(mol)
                    
    print(f"Loaded {len(training_smiles)} training mols. Found {len(fresh_mols)} fresh mols.")
    return fresh_mols

# Execute
fresh_mols = get_fresh_mols("sampleprior/Prior_100.smi", "all_unique_processed_dataset.smi")
# prerender=True helps when the widget cannot dynamically fetch images (common in VS Code)
mols2grid.display(fresh_mols, prerender=True)

Loaded 4323 training mols. Found 57 fresh mols.


[11:35:54] Can't kekulize mol.  Unkekulized atoms: 2 3 4 15 16 17 18
[11:35:54] SMILES Parse Error: unclosed ring for input: 'C1(=O)CCC(N2C(=O)c3c(c(C(=O)NC4(c4)nc[nH]5)CCCC4)CC2)C(=O)N1'
[11:35:54] SMILES Parse Error: extra close parentheses while parsing: O=C(NC(C1C2OC(C)(C)O3)=O)(CC)C(=O)NC1=O)c2
[11:35:54] SMILES Parse Error: check for mistakes around position 40:
[11:35:54] O3)=O)(CC)C(=O)NC1=O)c2
[11:35:54] ~~~~~~~~~~~~~~~~~~~~^
[11:35:54] SMILES Parse Error: Failed parsing SMILES 'O=C(NC(C1C2OC(C)(C)O3)=O)(CC)C(=O)NC1=O)c2' for input: 'O=C(NC(C1C2OC(C)(C)O3)=O)(CC)C(=O)NC1=O)c2'
[11:35:54] Can't kekulize mol.  Unkekulized atoms: 11 14 15
[11:35:54] SMILES Parse Error: extra open parentheses while parsing: C(N1CC(C(C)C)(c1nn(C)cc1C(=O)NC1CCC(=O)NC1=O)=O
[11:35:54] SMILES Parse Error: check for mistakes around position 2:
[11:35:54] C(N1CC(C(C)C)(c1nn(C)cc1C(=O)NC1CCC(=O)NC
[11:35:54] ~^
[11:35:54] SMILES Parse Error: Failed parsing SMILES 'C(N1CC(C(C)C)(c1nn(C)cc1C(=O)NC1CCC(=O)N

In [49]:
#change the strategy to fine tune from a large prior model
!python /home/haolan/SMILES-RNN/scripts/fine_tune.py \
-p /home/haolan/REINVENT4/Priors/reinvent.prior \
-i all_unique_processed_dataset.smi \
-o finetuneprior \
-s finetuneprior \
--model RNN \
--randomize

Device set to cuda
Loading smiles
Randomizing 4323 training smiles
100%|██████████████████████████████████████| 4323/4323 [00:11<00:00, 377.51it/s]
Returned 42407 randomized training smiles
{'I', '[H]', 'B', '[NH2+]', 'P', '[NH3+]'}
Beginning training
Epoch 1
100%|█████████████████████████████████████████| 331/331 [00:29<00:00, 11.30it/s]
Epoch 2
100%|█████████████████████████████████████████| 331/331 [00:26<00:00, 12.45it/s]
Epoch 3
100%|█████████████████████████████████████████| 331/331 [00:24<00:00, 13.50it/s]
Epoch 4
100%|█████████████████████████████████████████| 331/331 [00:22<00:00, 14.48it/s]
Epoch 5
100%|█████████████████████████████████████████| 331/331 [00:26<00:00, 12.43it/s]
Epoch 6
100%|█████████████████████████████████████████| 331/331 [00:29<00:00, 11.06it/s]
Epoch 7
100%|█████████████████████████████████████████| 331/331 [00:25<00:00, 13.20it/s]
Epoch 8
100%|█████████████████████████████████████████| 331/331 [00:25<00:00, 12.73it/s]
Epoch 9
100%|███████████████████████

In [51]:
log_dir = os.path.abspath("finetuneprior/tb_finetuneprior")
print(f"Launching TensorBoard for: {log_dir}")
%tensorboard --logdir "{log_dir}" --host localhost

Launching TensorBoard for: /mnt/d/blog_in_progress/smiles_rnn_test/finetuneprior/tb_finetuneprior


Reusing TensorBoard on port 6015 (pid 2021720), started 0:07:44 ago. (Use '!kill 2021720' to kill it.)

In [54]:
!python /home/haolan/SMILES-RNN/scripts/sample_model.py -p finetuneprior/Prior_finetuneprior_Epoch-10.ckpt \
-m RNN \
-o samplefinetuneprior/Prior_100.smi \
-n 100

Device set to cuda
Saving 100 smiles


In [56]:
fresh_mols = get_fresh_mols("samplefinetuneprior/Prior_100.smi", "all_unique_processed_dataset.smi")
mols2grid.display(fresh_mols, prerender=True)

Loaded 4323 training mols. Found 54 fresh mols.


[12:12:03] Explicit valence for atom # 21 N, 4, is greater than permitted
[12:12:03] SMILES Parse Error: unclosed ring for input: 'O=C1NC(=O)C(N2C(=O)c3cccc(-c4ccnn4-c4ccccc5)cc3C4C2)CC1'
